In [ ]:
%%configure -f
{"defaultLakehouse": {"name": "lh_fraud"}}

# 06 · Lakehouse layout — from `dbo` to medallion + purpose schemas (phase **finalize**)

| Schema | Role |
|---|---|
| `bronze` | Données brutes telles que livrées par la banque |
| `quality` | Contrôles qualité ligne à ligne et quarantaine |
| `silver` | Opérations nettoyées, conformées et jointes |
| `gold` | Caractéristiques, scores, file d'alertes, explications |
| `ml` | Évaluation et robustesse du modèle |
| `mart` | Schéma en étoile pour l'analyse |
| `serving` | Tables servies au rapport Power BI |
| `meta` | Catalogue de données et journal de migration |

`dbo` remains the default schema and ends empty. Phase **copy** creates the schemas, copies every table (Delta, V-Order) and verifies
row counts and columns; phase **finalize** runs only after the app, notebooks, copy job and Power BI model point to the new names:
it re-verifies, drops the `dbo` tables and reorganises `Files/`. Every step is written to `meta.migration_log`.

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

PHASE = "finalize"
TABLES = json.loads(r'''{"transactions_train": ["bronze", "transactions_train", "cj_bronze_load", "Opérations janv. 2025 → juil. 2026"], "transactions_test": ["bronze", "transactions_test", "cj_bronze_load", "Opérations août → sept. 2026 (labels détenus par le jury)"], "customers": ["bronze", "customers", "cj_bronze_load", "Clients (3 500)"], "cards": ["bronze", "cards", "cj_bronze_load", "Cartes"], "merchants": ["bronze", "merchants", "cj_bronze_load", "Commerçants et DAB"], "digital_events": ["bronze", "digital_events", "cj_bronze_load", "Événements de la banque digitale"], "fraud_labels_train": ["bronze", "fraud_labels_train", "cj_bronze_load", "Fraudes signalées (période d'entraînement)"], "legacy_rule_alerts_train": ["bronze", "legacy_rule_alerts_train", "cj_bronze_load", "Alertes du moteur de règles actuel"], "v_transactions_all": ["bronze", "v_transactions_all", "cj_bronze_load", "Vue source : toutes les opérations"], "v_transactions_train_labeled": ["bronze", "v_transactions_train_labeled", "cj_bronze_load", "Vue source : opérations labellisées"], "bronze_ingestion_log": ["bronze", "ingestion_log", "05_quality_star", "Journal des chargements (lignes, colonnes, empreinte du schéma)"], "dq_rules": ["quality", "rules", "05_quality_star", "41 règles documentées (ERROR / WARNING)"], "dq_issues": ["quality", "issues", "05_quality_star", "Une ligne par anomalie : table, enregistrement, règle, valeur"], "dq_row_status": ["quality", "row_status", "05_quality_star", "Statut de chaque enregistrement : OK / WARNING / REJECTED"], "dq_summary": ["quality", "summary", "05_quality_star", "Taux d'échec par règle"], "quarantine_transactions": ["quality", "quarantine_transactions", "05_quality_star", "Opérations rejetées"], "quarantine_customers": ["quality", "quarantine_customers", "05_quality_star", "Clients rejetés"], "quarantine_cards": ["quality", "quarantine_cards", "05_quality_star", "Cartes rejetées"], "quarantine_merchants": ["quality", "quarantine_merchants", "05_quality_star", "Commerçants rejetés"], "quarantine_digital_events": ["quality", "quarantine_digital_events", "05_quality_star", "Événements rejetés"], "silver_transactions": ["silver", "transactions", "01_silver_features", "Opérations nettoyées + client + carte + commerçant"], "gold_features": ["gold", "features", "01_silver_features", "73 caractéristiques calculées sur le passé uniquement"], "gold_scores": ["gold", "scores", "02_score_explain", "Score du modèle de chaque opération"], "gold_alert_queue": ["gold", "alert_queue", "02_score_explain", "File d'alertes (top 1 % par période)"], "gold_alert_explanations": ["gold", "alert_explanations", "02_score_explain", "Facteurs SHAP et règles par alerte"], "gold_model_metrics": ["ml", "model_metrics", "02_score_explain", "Métriques de validation"], "gold_model_leaderboard": ["ml", "model_leaderboard", "02_score_explain", "Comparaison des modèles"], "gold_model_importance": ["ml", "model_importance", "02_score_explain", "Importance des variables"], "gold_model_recall_by_type": ["ml", "model_recall_by_type", "02_score_explain", "Rappel par typologie de fraude"], "gold_model_ci": ["ml", "model_ci", "04_robustness", "Intervalles de confiance (bootstrap par jour)"], "gold_model_robustness": ["ml", "model_robustness", "04_robustness", "Backtest glissant sur 4 périodes"], "gold_model_robustness_types": ["ml", "model_robustness_types", "04_robustness", "Backtest par typologie"], "dim_date": ["mart", "dim_date", "05_quality_star", "Calendrier (jours fériés, paie, période du modèle)"], "dim_heure": ["mart", "dim_heure", "05_quality_star", "Heures et tranches horaires"], "dim_geo": ["mart", "dim_geo", "05_quality_star", "Ville → gouvernorat → région → pays"], "dim_client": ["mart", "dim_client", "05_quality_star", "Profil client (sans genre)"], "dim_merchant": ["mart", "dim_merchant", "05_quality_star", "Commerçants"], "dim_channel": ["mart", "dim_channel", "05_quality_star", "Type d'opération × canal × mode de saisie"], "dim_card": ["mart", "dim_card", "05_quality_star", "Cartes"], "fact_transactions": ["mart", "fact_transactions", "05_quality_star", "Fait : opérations + score + alerte"], "fact_digital_events": ["mart", "fact_digital_events", "05_quality_star", "Fait : événements de l'application"], "pbi_alerts": ["serving", "pbi_alerts", "03e_powerbi_dashboard", "Power BI · alerts"], "pbi_compare": ["serving", "pbi_compare", "03e_powerbi_dashboard", "Power BI · compare"], "pbi_factors": ["serving", "pbi_factors", "03e_powerbi_dashboard", "Power BI · factors"], "pbi_importance": ["serving", "pbi_importance", "03e_powerbi_dashboard", "Power BI · importance"], "pbi_kpi": ["serving", "pbi_kpi", "03e_powerbi_dashboard", "Power BI · kpi"], "pbi_leaderboard": ["serving", "pbi_leaderboard", "03e_powerbi_dashboard", "Power BI · leaderboard"], "pbi_recall_type": ["serving", "pbi_recall_type", "03e_powerbi_dashboard", "Power BI · recall_type"], "pbi_recall_wide": ["serving", "pbi_recall_wide", "03e_powerbi_dashboard", "Power BI · recall_wide"]}''')
LAYERS = json.loads(r'''{"bronze": ["Données brutes telles que livrées par la banque", "Raw data as delivered by the bank"], "quality": ["Contrôles qualité ligne à ligne et quarantaine", "Row-level data quality and quarantine"], "silver": ["Opérations nettoyées, conformées et jointes", "Cleaned, conformed and joined operations"], "gold": ["Caractéristiques, scores, file d'alertes, explications", "Features, scores, alert queue, explanations"], "ml": ["Évaluation et robustesse du modèle", "Model evaluation and robustness"], "mart": ["Schéma en étoile pour l'analyse", "Star schema for analysis"], "serving": ["Tables servies au rapport Power BI", "Tables served to the Power BI report"], "meta": ["Catalogue de données et journal de migration", "Data catalog and migration log"]}''')
FILES = json.loads(r'''{"app_export": "exports/app_snapshot", "submission": "exports/submission", "app_feedback": "feedback/analyst_decisions", "robustness": "ml/robustness"}''')
RUN = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
log = []

def note(step, obj, status, detail=""):
    log.append((RUN, PHASE, step, obj, status, str(detail)[:500]))
    print(f"{status:8s} {step:10s} {obj} {detail}")

def exists(name):
    try:
        spark.table(name).limit(0).collect()
        return True
    except Exception:
        return False

## Schemas

In [ ]:
for s in list(LAYERS):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")
    note("schema", s, "OK")

## Finalize: re-verify, drop `dbo`, reorganise `Files/`

In [ ]:
for old, (schema, new, producer, desc) in TABLES.items():
    src, dst = f"dbo.{old}", f"{schema}.{new}"
    if not exists(src):
        note("drop", src, "SKIP", "already gone")
        continue
    a, b = spark.table(src).count(), spark.table(dst).count()
    if a != b:
        note("drop", src, "KEPT", f"{a} vs {b} rows — not dropped")
        continue
    spark.sql(f"DROP TABLE {src}")
    note("drop", src, "OK", f"{b} rows now only in {dst}")
assert not [l for l in log if l[4] == "KEPT"], "some dbo tables kept"

def mv(a, b):
    try:
        if notebookutils.fs.exists(a):
            notebookutils.fs.mkdirs("/".join(b.split("/")[:-1]))
            notebookutils.fs.mv(a, b, True)
            note("files", f"{a} -> {b}", "OK")
        else:
            note("files", a, "SKIP", "absent")
    except Exception as exc:
        note("files", a, "ERROR", exc)
for old, new in FILES.items():
    mv(f"Files/{old}", f"Files/{new}")
for d in ["landing/partner_feeds", "exports", "feedback", "ml"]:
    notebookutils.fs.mkdirs(f"Files/{d}")
notebookutils.fs.put("Files/README.md", """# lh_fraud · Files
- `landing/partner_feeds/` — files received from other banks, before validation (page « Nouvelle source de données »)
- `exports/app_snapshot/` — Gold snapshot for the hosted demo app
- `exports/submission/` — score file for the jury (brief §14)
- `feedback/analyst_decisions/` — analyst decisions written back by the app (hash-chained journal)
- `ml/robustness/` — backtest summary and runtime versions (notebook 04)
Tables: see meta.table_catalog (bronze · quality · silver · gold · ml · mart · serving · meta).
""", True)
note("files", "Files/README.md", "OK")

## Catalog

In [ ]:
rows = []
for old, (schema, new, producer, desc) in TABLES.items():
    name = f"{schema}.{new}"
    n = spark.table(name).count() if exists(name) else None
    cols = len(spark.table(name).columns) if exists(name) else None
    rows.append((schema, new, name, LAYERS[schema][1], desc, producer, f"dbo.{old}", n, cols, RUN))
spark.createDataFrame(rows, "schema_name string, table_name string, full_name string, layer_role string, description string, "
                            "producer string, previous_name string, row_count long, column_count int, cataloged_at string") \
     .write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("meta.table_catalog")
spark.createDataFrame(log, "run_ts string, phase string, step string, object string, status string, detail string") \
     .write.mode("append").format("delta").saveAsTable("meta.migration_log")
display(spark.table("meta.table_catalog").orderBy("schema_name", "table_name"))
print("schemas:", [r.namespace if hasattr(r, "namespace") else r[0] for r in spark.sql("SHOW SCHEMAS").collect()])